# Project TuneCraft — LoRA Fine-Tuning of TinyLlama-1.1B
**Orion AI Labs · LLM Optimization Task Force**

Pipeline: base-model evaluation → dataset engineering → LoRA injection → SFT training → adapter persistence → base-vs-tuned evaluation.

- Base model: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`
- Framework: Hugging Face TRL + PEFT (LoRA)
- Dataset: `databricks/databricks-dolly-15k`
- Runtime: Google Colab, **T4 GPU**

---
## Part 0 — Setup

In [1]:
# Installs pinned, known-good versions ONCE per fresh runtime, then restarts the runtime.
# On later runs (or after a restart) the marker file exists and this cell does nothing.
import os, sys, subprocess

READY = "/tmp/.tunecraft_deps_ok"        # /tmp is wiped on a new runtime, kept on 'Restart session'
if not os.path.exists(READY):
    # Colab's preinstalled torchao (0.10) is too old for PEFT -> remove it (not needed: no quantization)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.17.0", "trl==1.13.0", "peft==0.21.0",
                    "accelerate==1.15.0", "datasets"], check=True)
    open(READY, "w").write("ok")
    print("Dependencies installed. Runtime is restarting - click Runtime > Run all once more.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed.")

Dependencies already installed.


In [2]:
# Install dependencies, then Runtime > Restart session once before continuing.
!pip install -U transformers trl peft datasets accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.1 MB/s eta 0:00:00


In [3]:
import os
# Colab sometimes ships with these set; force online mode so model/dataset downloads work
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"
os.environ["HF_DATASETS_OFFLINE"] = "0"

import gc, math, inspect, shutil
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
PyTorch 2.11.0+cu128 | GPU: Tesla T4


### Utility functions (provided)

In [12]:
def clear_memory():
    """Free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


def free_vars(*names):
    """Delete global variables (if they exist) and release GPU memory -> makes cells re-runnable."""
    for n in names:
        globals().pop(n, None)
    clear_memory()

def print_trainable_params(model):
    """Display trainable vs total parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:>12,}  ({100 * trainable / total:.4f}%)")
    print(f"Total parameters:     {total:>12,}")


def load_base_model():
    """Always returns a FRESH, unmodified base model (LoRA wrapping edits a model in place)."""
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")


def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """Generate a response from a model given a prompt string."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

---
## Part 1 — Configuration
### Task 1: Define all hyperparameters


Set the following configuration variables:

| Variable | Value |
|---|---|
| `MODEL_NAME` | `"TinyLlama/TinyLlama-1.1B-Chat-v1.0"` |
| `DATASET_NAME` | `"databricks/databricks-dolly-15k"` |
| `NUM_TRAIN_SAMPLES` | `1000` |
| `MAX_SEQ_LENGTH` | `512` |
| `LORA_R` | `8` |
| `LORA_ALPHA` | `16` |
| `LORA_DROPOUT` | `0.05` |
| `LORA_TARGET_MODULES` | `["q_proj", "v_proj"]` (attention only) |
| `LEARNING_RATE` | `2e-4` |
| `NUM_EPOCHS` | `1` |
| `BATCH_SIZE` | `4` |
| `GRAD_ACCUM_STEPS` | `4` |
| `OUTPUT_DIR` | `"./sft-lora-tinyllama"` |

In [5]:
MODEL_NAME   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_NAME = "databricks/databricks-dolly-15k"

NUM_TRAIN_SAMPLES = 1000
MAX_SEQ_LENGTH    = 512

LORA_R              = 8
LORA_ALPHA          = 16
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]   # attention only

LEARNING_RATE    = 2e-4
NUM_EPOCHS       = 1
BATCH_SIZE       = 4
GRAD_ACCUM_STEPS = 4

OUTPUT_DIR = "./sft-lora-tinyllama"
SEED       = 42
set_seed(SEED)

---
## Part 2 — Load Model and Tokenizer
### Task 2: Load the tokenizer and base model

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Pad token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Pad token: </s>


In [7]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,      # (older transformers versions: torch_dtype=)
    device_map="auto",
)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total parameters: 1,100,048,384


---
## Part 3 — Test Base Model
### Task 3: Generate responses from the base model
A fixed seed is set before every generation so base vs fine-tuned differences come from the weights, not sampling noise.

In [14]:
test_prompts = [
    "Explain what a neural network is in 2-3 sentences.",
    "What are three tips for better sleep?",
]

free_vars("model", "lora_model", "trainer")      # guarantees an un-wrapped base model, even on re-run
base_model = load_base_model().eval()

base_responses = []
for i, prompt in enumerate(test_prompts):
    set_seed(SEED + i)
    response = generate_response(base_model, tokenizer, prompt)
    base_responses.append(response)
    print("=" * 80)
    print("PROMPT:", prompt)
    print("\nBASE RESPONSE:\n" + response)

free_vars("base_model")

GPU allocated: 2.06 GB
GPU reserved:  2.09 GB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: Explain what a neural network is in 2-3 sentences.

BASE RESPONSE:
A neural network is a mathematical model that mimics the structure and function of the human brain. It consists of a set of interconnected units, or neurons, that receive input from other neurons through synapses, process the input, and output a response or action. Neural networks can be trained to perform a variety of tasks, such as image recognition, speech recognition, and decision making, by learning from data and adjusting their weights and biases accordingly. The main advantage of neural networks is their ability to learn from and adapt to new data, making them useful in a wide range of applications, including robotics, artificial intelligence, and healthcare.
PROMPT: What are three tips for better sleep?

BASE RESPONSE:
Here are three tips for better sleep:

1. Create a sleep-conducive environment: Make sure your bedroom is cool, dark, and quiet. Avoid screens and noise during the night.

2. Establish a s

---
## Part 4 — Prepare the Dataset
### Task 4: Load and format the Dolly dataset

In [15]:
dataset = load_dataset(DATASET_NAME, split="train")
print(dataset)
print(dataset[0])

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [16]:
def format_dolly_to_chat(example):
    """Convert a Dolly row into the chat 'messages' format."""
    instruction = example["instruction"]
    context = example["context"]

    if context and context.strip():
        user_content = instruction + "\n" + context
    else:
        user_content = instruction

    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example["response"]},
        ]
    }

shuffled = dataset.shuffle(seed=SEED)

# 1,000 training samples (only the 'messages' column is kept)
train_dataset = (
    shuffled
    .select(range(NUM_TRAIN_SAMPLES))
    .map(format_dolly_to_chat, remove_columns=dataset.column_names)
)

# 100 held-out samples (never seen in training) used for quantitative evaluation in Part 8
eval_dataset = (
    shuffled
    .select(range(NUM_TRAIN_SAMPLES, NUM_TRAIN_SAMPLES + 100))
    .map(format_dolly_to_chat, remove_columns=dataset.column_names)
)

print("Formatted train size:", len(train_dataset))
print("Held-out eval size:  ", len(eval_dataset))
print("\nExample:")
print(train_dataset[0])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Formatted train size: 1000
Held-out eval size:   100

Example:
{'messages': [{'content': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?', 'role': 'user'}, {'content': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree', 'role': 'assistant'}]}


---
## Part 5 — Configure LoRA
### Task 5: Create the LoRA config and inspect trainable parameters

In [17]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

# Wrap a FRESH model just to inspect it (get_peft_model edits the model in place)
free_vars("model", "lora_model", "trainer")
model = load_base_model()
lora_model = get_peft_model(model, lora_config)
print_trainable_params(lora_model)

print("\nLoRA-wrapped q_proj of layer 0:")
print(lora_model.base_model.model.model.layers[0].self_attn.q_proj)

GPU allocated: 0.01 GB
GPU reserved:  2.05 GB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Trainable parameters:    1,126,400  (0.1023%)
Total parameters:     1,101,174,784

LoRA-wrapped q_proj of layer 0:
lora.Linear(
  (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=2048, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=2048, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)


---
## Part 6 — Train
### Task 6: Set up SFTTrainer and run training

In [18]:
# 1) Delete the PEFT-wrapped model and reload a fresh base model
#    (SFTTrainer applies LoRA itself through `peft_config`)

free_vars("lora_model", "model", "trainer")
model = load_base_model()
print("Fresh base model loaded")


GPU allocated: 0.01 GB
GPU reserved:  2.05 GB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Fresh base model loaded


In [19]:
# 2) Precision: T4 (compute capability 7.5) has no native bf16 -> use fp16 there.
#    torch.cuda.is_bf16_supported() can return True on a T4 via slow emulation, so check capability.
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
print("Using", "bf16" if use_bf16 else "fp16")

# warmup ratio 0.03: newer transformers takes a float <1 in `warmup_steps` as a ratio,
# older versions use `warmup_ratio`. Pick whichever this install supports.
_params = inspect.signature(SFTConfig).parameters
warmup_kwargs = {"warmup_ratio": 0.03} if "warmup_ratio" in _params else {"warmup_steps": 0.03}

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)     # start clean on every re-run

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=use_bf16,
    fp16=not use_bf16,
    max_length=MAX_SEQ_LENGTH,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    **warmup_kwargs,
)

Using fp16


In [20]:
# 3) Trainer: uses the 1,000-sample `train_dataset` (NOT the full 15k dataset)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
print_trainable_params(trainer.model)

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4117 > 2048). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Trainable parameters:    1,126,400  (0.1023%)
Total parameters:     1,101,174,784


In [21]:
# 4) Train
train_result = trainer.train()

last_logged = [l["loss"] for l in trainer.state.log_history if "loss" in l][-1]
print(f"Average training loss: {train_result.training_loss:.4f}")
print(f"Last logged loss:      {last_logged:.4f}")

Step,Training Loss
10,2.109733
20,1.878646
30,1.762211
40,1.737146
50,1.782263
60,1.725337


Average training loss: 1.8320
Last logged loss:      1.7253


---
## Part 7 — Save the Adapter
### Task 7: Save the LoRA adapter and print its size

In [22]:
ADAPTER_PATH = f"{OUTPUT_DIR}/final-adapter"

trainer.model.save_pretrained(ADAPTER_PATH)   # saves ONLY the LoRA weights
tokenizer.save_pretrained(ADAPTER_PATH)

files = {f: os.path.getsize(os.path.join(ADAPTER_PATH, f)) for f in os.listdir(ADAPTER_PATH)}
for f, s in sorted(files.items()):
    print(f"{f:35s} {s / 1024**2:8.2f} MB")

weights_mb = files.get("adapter_model.safetensors", 0) / 1024**2
total_mb = sum(files.values()) / 1024**2
print(f"\nAdapter weights:        {weights_mb:.2f} MB")
print(f"Whole folder (w/ tokenizer): {total_mb:.2f} MB   vs ~2,200 MB for the full model")

README.md                               0.00 MB
adapter_config.json                     0.00 MB
adapter_model.safetensors               4.31 MB
chat_template.jinja                     0.00 MB
tokenizer.json                          3.45 MB
tokenizer_config.json                   0.00 MB

Adapter weights:        4.31 MB
Whole folder (w/ tokenizer): 7.77 MB   vs ~2,200 MB for the full model


---
## Part 8 — Evaluate
### Task 8: Compare base vs fine-tuned responses

In [30]:
# Use the trainer's model if it is still in memory; otherwise (e.g. after a restart) load the saved adapter.
if "trainer" in globals():
    ft_model = trainer.model
else:
    ft_model = PeftModel.from_pretrained(load_base_model(), ADAPTER_PATH)
ft_model.eval()


ft_responses = []
for i, prompt in enumerate(test_prompts):
    set_seed(SEED + i)                            # same seeds as the base run
    ft_responses.append(generate_response(ft_model, tokenizer, prompt))

# ============================================================
# QUALITATIVE EVALUATION
# BASE MODEL vs FINE-TUNED MODEL
# ============================================================

print("=" * 100)
print("QUALITATIVE EVALUATION: BASE MODEL vs FINE-TUNED (LoRA)")
print("=" * 100)

for prompt, base, ft in zip(test_prompts, base_responses, ft_responses):
    print("=" * 100)
    print("PROMPT:", prompt)
    print("-" * 100)
    print("BASE MODEL:\n" + base)
    print("-" * 100)
    print("FINE-TUNED (LoRA):\n" + ft)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUALITATIVE EVALUATION: BASE MODEL vs FINE-TUNED (LoRA)
PROMPT: Explain what a neural network is in 2-3 sentences.
----------------------------------------------------------------------------------------------------
BASE MODEL:
A neural network is a mathematical model that mimics the structure and function of the human brain. It consists of a set of interconnected units, or neurons, that receive input from other neurons through synapses, process the input, and output a response or action. Neural networks can be trained to perform a variety of tasks, such as image recognition, speech recognition, and decision making, by learning from data and adjusting their weights and biases accordingly. The main advantage of neural networks is their ability to learn from and adapt to new data, making them useful in a wide range of applications, including robotics, artificial intelligence, and healthcare.
-------------------------------------------------------------------------------------------------

In [32]:
# ============================================================
# QUALITATIVE EVALUATION
# BASE MODEL vs FINE-TUNED (LoRA)
# ============================================================

import pandas as pd
from IPython.display import display

comparison_df = pd.DataFrame({
    "PROMPT": test_prompts,
    "BASE MODEL": base_responses,
    "FINE-TUNED (LoRA)": ft_responses
})

print("BASE MODEL vs FINE-TUNED (LoRA) — QUALITATIVE COMPARISON")
display(comparison_df)

BASE MODEL vs FINE-TUNED (LoRA) — QUALITATIVE COMPARISON


,PROMPT,BASE MODEL,FINE-TUNED (LoRA)
0,Explain what a neural network is in 2-3 senten...,A neural network is a mathematical model that ...,A neural network is a mathematical model that ...
1,What are three tips for better sleep?,Here are three tips for better sleep:\n\n1. Cr...,1. Stick to a consistent sleep routine.\n2. Ke...


In [25]:
# Quantitative check on 100 HELD-OUT Dolly examples (lower loss / perplexity = better).
# Loss is computed over the full chat text (prompt + answer).
@torch.no_grad()
def avg_loss(m, ds):
    m.eval()
    losses = []
    for ex in ds:
        text = tokenizer.apply_chat_template(ex["messages"], tokenize=False)
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(m.device)
        with torch.autocast("cuda", dtype=torch.float16):
            out = m(**enc, labels=enc["input_ids"])
        losses.append(out.loss.item())
    return sum(losses) / len(losses)

with ft_model.disable_adapter():            # adapter off  -> original base model
    base_loss = avg_loss(ft_model, eval_dataset)
ft_loss = avg_loss(ft_model, eval_dataset)  # adapter on   -> fine-tuned model

print(f"Base model  : loss {base_loss:.4f} | perplexity {math.exp(base_loss):.2f}")
print(f"Fine-tuned  : loss {ft_loss:.4f} | perplexity {math.exp(ft_loss):.2f}")
print(f"Loss reduction: {100 * (base_loss - ft_loss) / base_loss:.1f}%")

Base model  : loss 2.4685 | perplexity 11.80
Fine-tuned  : loss 1.7306 | perplexity 5.64
Loss reduction: 29.9%


In [26]:
# Persistence check: fresh base model + SAVED adapter from disk
reloaded = PeftModel.from_pretrained(load_base_model(), ADAPTER_PATH).eval()
set_seed(SEED)
print(generate_response(reloaded, tokenizer, test_prompts[0]))
free_vars("reloaded")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A neural network is a mathematical model that mimics the structure and function of the human brain. It consists of a set of layers of neurons connected by synapses, each neuron receiving input from other neurons through synaptic connections. The network is trained using a training dataset to learn the relationships between input and output.
GPU allocated: 2.08 GB
GPU reserved:  2.13 GB


---
## Part 9 — Share / reuse the adapter (optional)

In [27]:
# # A) Zip the adapter and download it (Colab only)
# shutil.make_archive("final-adapter", "zip", ADAPTER_PATH)
# try:
#     from google.colab import files as colab_files
#     colab_files.download("final-adapter.zip")
# except ImportError:
#     print("Not on Colab - zip saved as final-adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# # B) Push to the Hugging Face Hub so anyone can load it. Needs a WRITE token saved in Colab Secrets as HF_TOKEN.
# PUSH_TO_HUB = False
# HUB_REPO    = "YOUR_USERNAME/tinyllama-dolly-lora"

# if PUSH_TO_HUB:
#     from huggingface_hub import login
#     from google.colab import userdata
#     login(userdata.get("HF_TOKEN"))
#     ft_model.push_to_hub(HUB_REPO)
#     tokenizer.push_to_hub(HUB_REPO)
#     print("Pushed to https://huggingface.co/" + HUB_REPO)

### Using the adapter elsewhere
```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok   = AutoTokenizer.from_pretrained("YOUR_USERNAME/tinyllama-dolly-lora")   # or a local folder
base  = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, "YOUR_USERNAME/tinyllama-dolly-lora")
```

---
## Done
Note in your write-up: (1) trainable params ≈ 0.10% of the model, (2) adapter size in MB vs the full model, (3) held-out loss/perplexity before vs after, (4) qualitative differences in the responses above (style, length, directness).